# Make re-execution safe; pick the right tool

> 📓 *Companion to* **Modern Agentic AI Engineer** *· Ch 31 §31.4–31.5, §31.7 · type: walkthrough*

**The promise:** you will make a Celery task **idempotent** so the `acks_late` double-run can't corrupt data, then learn to *choose* between a task queue, a durable-execution engine, and a DAG orchestrator — and see exactly why an agent's non-determinism breaks naive replay, and how `record_activity` fixes it.

## 🧠 Why this matters

Notebook `31-01` left you with a sharp edge: `acks_late` gives you reliable delivery, but at the cost of **at-least-once** — a task can run more than once. If that task charges a card or sends an email, a re-run *doubles* the side effect. The cure is **idempotency**: design every side-effecting task so that running it twice has the same effect as running it once.

Then comes a higher-altitude judgment. Celery runs *tasks*; some work is a *workflow* — multi-step, long-running, must survive crashes and resume exactly where it left off. **Durable execution** engines (Temporal-style) give you that via **event-sourced replay** — but point replay at an *agent* and a subtle, production-only bug appears: the LLM returns something different on re-call, and the replay diverges from its own recorded history. Knowing the fix (treat each model/tool call as a recorded *activity*) and knowing *which* tool fits *which* shape of work is what separates a senior architect from someone who hand-rolls a fragile multi-day workflow in Celery.

## Objectives & prereqs

**By the end you can:**
- Make a Celery task **idempotent** with an idempotency key + an "already done?" ledger check, so a double-run is "effectively once."
- Choose between **Celery**, **durable execution** (Temporal), and a **DAG orchestrator** (Prefect/Airflow) for a given workload.
- Explain why **durable-execution replay breaks on an agent** and apply the `record_activity` fix (recorded results + per-step idempotency keys).

**Prereqs:** `31-01` (you have the `acks_late` double-run in hand); Ch 29 (idempotency keys, the outbox); Ch 16 (the agent loop) is *referenced*, not required.

**Runs free & offline.** Everything is pure local Python — an in-memory ledger and a mock "activity log." The agent/model step is **mocked** (a canned plan/result) so replay and idempotency stay deterministic. No services, no API key. Seeded RNG throughout.

In [ ]:
# Setup — imports, env, and the MOCK switch.
import os
import random
from dataclasses import dataclass, field

from dotenv import load_dotenv

load_dotenv()

# MOCK=1 (default): the agent/model step returns a CANNED plan/result so replay and
# idempotency are deterministic. MOCK=0 would call a real model for the 'plan' step
# (and the divergence demo would become genuinely non-deterministic — that's the point).
MOCK = os.getenv("COMPANION_MOCK", "1") == "1"

# Seed so the 'non-deterministic' model and any RNG are reproducible WITHIN a run,
# while we still simulate divergence explicitly where the lesson needs it.
random.seed(2916)

print("MOCK mode:", MOCK, "— canned agent step, offline" if MOCK else "— LIVE model on the plan step")

## 🔮 Predict: run a non-idempotent task twice

Recall the `acks_late` reality — the same message can be delivered twice. Below, a "send welcome email" task appends to an outbox every time it runs, with **no guard**. We invoke it twice for the same user, simulating the crash-after-finish re-delivery.

**Predict:** how many emails land in the outbox — one or two? Decide, then run.

In [ ]:
EMAIL_OUTBOX = []  # pretend this is the mail provider's send log


def send_welcome_email(user_id: str):
    """⚠️ Non-idempotent: every call sends another email."""
    EMAIL_OUTBOX.append({"to": user_id, "subject": "Welcome!"})
    return {"sent_to": user_id}


# The acks_late re-run: same logical task delivered twice.
send_welcome_email("user-7")
send_welcome_email("user-7")

print("emails sent:", len(EMAIL_OUTBOX))
for e in EMAIL_OUTBOX:
    print("  →", e)
print("\nuser-7 got the welcome email TWICE — the duplicate is the bug.")

**What you just saw.** Two emails for one signup. In a real system that's an angry customer at best, a double charge at worst. The re-run isn't a freak event — it's the *designed* behaviour of an at-least-once queue. So the task, not the queue, has to change.

## Fix it: an idempotency key + an "already done?" check

The Ch 29 pattern: give each logical operation an **idempotency key**, and keep a **ledger** of keys already completed. Before doing the side effect, check the ledger; after, record the key (with its result). A second execution finds the key and returns the *recorded* outcome — a no-op. The effect on the world is **effectively once**, even though the task ran twice.

In [ ]:
# The ledger: idempotency_key -> recorded result. In production this is a DB row with a
# UNIQUE constraint on the key (Ch 29/Ch 30), not a dict — but the logic is identical.
LEDGER = {}
SAFE_OUTBOX = []


def send_welcome_email_idempotent(user_id: str, idempotency_key: str):
    if idempotency_key in LEDGER:
        # Already done — return the recorded outcome WITHOUT repeating the side effect.
        return {**LEDGER[idempotency_key], "replayed": True}
    result = {"to": user_id, "subject": "Welcome!"}
    SAFE_OUTBOX.append(result)            # the side effect, performed exactly once
    LEDGER[idempotency_key] = result      # record the key + result atomically (Ch 29)
    return {**result, "replayed": False}


# Same idempotency key for the same logical signup → the re-run is a no-op.
key = "signup:user-7"
first = send_welcome_email_idempotent("user-7", key)
second = send_welcome_email_idempotent("user-7", key)   # the acks_late re-delivery

print("first run :", first)
print("second run:", second)
print("emails actually sent:", len(SAFE_OUTBOX))
assert len(SAFE_OUTBOX) == 1, "idempotency failed — the side effect ran twice"
print("PASS — ran twice, sent once: 'effectively once'")

**What you just saw.** Two invocations, **one** email. The second call short-circuits on the ledger and returns the recorded result flagged `replayed=True`. That is the whole game: the queue stays at-least-once (reliable), and the *task* makes re-execution harmless. Every side-effecting task in your `workers/` directory needs this shape.

## Tool fit: queue vs durable execution vs DAG

Celery is the heavyweight, but it isn't the only shape of "do work later." Match the tool to the *shape* of the work — using a queue to hand-roll a resumable, multi-day workflow is a common path to a fragile mess.

In [ ]:
TOOL_FIT = {
    "Celery / task queue": {
        "use_when": "fire-and-forget or simple periodic tasks (reindex, send email, nightly job)",
        "field":    "Dramatiq (simpler), RQ (Redis-only), arq (asyncio/FastAPI-native)",
    },
    "Durable execution (Temporal)": {
        "use_when": "long, stateful, RESUMABLE process with waits & human-in-the-loop steps",
        "field":    "workflow code that survives crashes via event-sourced replay",
    },
    "DAG orchestrator (Prefect/Airflow)": {
        "use_when": "a SCHEDULED DAG of data transforms (nightly embedding refresh, ETL)",
        "field":    "scheduling, retries, lineage, and visibility over batch pipelines",
    },
}
for tool, info in TOOL_FIT.items():
    print(f"{tool}")
    print(f"    use when : {info['use_when']}")
    print(f"    field    : {info['field']}\n")


def pick_tool(*, resumable: bool, waits_or_human: bool, is_dag: bool) -> str:
    if is_dag:
        return "DAG orchestrator (Prefect/Airflow)"
    if resumable and waits_or_human:
        return "Durable execution (Temporal)"
    return "Celery / task queue"


print("nightly embedding refresh :", pick_tool(resumable=False, waits_or_human=False, is_dag=True))
print("send a welcome email      :", pick_tool(resumable=False, waits_or_human=False, is_dag=False))
print("3-day approval workflow   :", pick_tool(resumable=True,  waits_or_human=True,  is_dag=False))

## 🧠 Durable execution = event-sourced replay

Here's the mental model that makes the next pitfall click. A durable-execution engine records the workflow's **history** of step results. After a crash it **re-runs your workflow function from the top**, but instead of re-executing each step it *feeds in the recorded result*. Your code reaches the crash point with all prior state rebuilt, then continues. The trick has exactly one load-bearing assumption: **your workflow is deterministic** — same inputs, same path.

In [ ]:
# A tiny event-sourced replayer. `history` is the recorded result of each step by name.
@dataclass
class Workflow:
    history: dict = field(default_factory=dict)   # step_name -> recorded result
    live_calls: list = field(default_factory=list)  # which steps actually executed

    def step(self, name, fn):
        """Run fn the first time; on replay, return the recorded result (don't re-run)."""
        if name in self.history:
            return self.history[name]            # REPLAY: recorded value, fn not called
        result = fn()                            # LIVE: execute once and record
        self.live_calls.append(name)
        self.history[name] = result
        return result


# A deterministic workflow replays perfectly: re-running from the top calls nothing new.
wf = Workflow()
wf.step("fetch", lambda: "corpus kb-42")
wf.step("count", lambda: 1200)
print("history after first run:", wf.history)

wf.live_calls.clear()
# ...crash, then replay from the top with the SAME history:
wf.step("fetch", lambda: "corpus kb-42")
wf.step("count", lambda: 1200)
print("steps executed on replay:", wf.live_calls, "(empty → everything came from history)")

## ⚠️ Pitfall: point replay at an agent and it diverges

A model call breaks the determinism assumption *flatly*. Re-call the LLM on replay and it returns a *different* plan, tool, or argument — so the live execution diverges from its own recorded history, and replay corrupts the very state it was meant to restore. Below, the "plan" step is a model call that returns something different on the replay re-call.

In [ ]:
# A 'model' whose output drifts between calls — the realistic agent behaviour.
_PLAN_RESPONSES = iter([
    "plan: [search_docs, summarize, post_slack]",   # original run
    "plan: [search_docs, rerank, summarize]",        # re-call on replay — DIFFERENT
])


def model_plan():
    """Mock LLM: non-deterministic across calls (MOCK=1 makes the drift explicit)."""
    return next(_PLAN_RESPONSES)


# NAIVE durable workflow: the plan step is re-executed on replay (treated as ordinary code).
original_plan = model_plan()                 # first, live run
print("recorded history says the plan was:", original_plan)

# ...crash, replay re-runs the workflow top-down and RE-CALLS the model:
replayed_plan = model_plan()                 # the bug: a fresh, different answer
print("replay re-called the model and got:", replayed_plan)

print("\ndiverged?", original_plan != replayed_plan,
      "— live execution now follows a DIFFERENT path than its recorded history → corruption")

## The fix: each model/tool call is a *recorded activity*

Stop treating model and tool calls as ordinary code. Make each one a **recorded activity** whose *result* is persisted the first time it runs. On replay, the engine returns the recorded output — **the model is never re-called** — so the run retraces its exact original path; non-determinism is captured once and frozen. This is the book's `record_activity` shape:

```python
plan   = yield record_activity("plan", lambda: model.complete(goal))
result = yield record_activity("call", lambda: tool.run(plan.action),
                               idempotency_key=f"{run_id}:{step}")
```

Recorded results make the run *deterministic on replay*; the `idempotency_key` on every side-effecting step makes its effects on the world **exactly-once** when it does run.

In [ ]:
# A fresh drifting model + a side-effecting tool with an idempotency ledger.
_PLAN_RESPONSES_2 = iter([
    "plan: [search_docs, summarize, post_slack]",
    "plan: [search_docs, rerank, summarize]",   # would be returned IF re-called — it won't be
])
SIDE_EFFECTS = []          # e.g. tickets filed / slack posts
EFFECT_LEDGER = {}          # idempotency_key -> recorded outcome


def record_activity(history, name, fn, idempotency_key=None):
    """Recorded-activity primitive: persist the result once; replay returns it (no re-call).
    For side-effecting activities, an idempotency_key pins the effect to exactly-once."""
    if name in history:
        return history[name]                       # REPLAY: recorded result, fn NOT called
    if idempotency_key is not None and idempotency_key in EFFECT_LEDGER:
        result = EFFECT_LEDGER[idempotency_key]     # half-finished resume: reuse the outcome
    else:
        result = fn()                               # LIVE: run once
        if idempotency_key is not None:
            EFFECT_LEDGER[idempotency_key] = result
    history[name] = result
    return result


def post_slack(text):
    SIDE_EFFECTS.append(text)   # the real-world side effect
    return {"posted": text}


run_id, history = "run-1", {}


def run_workflow(history):
    plan = record_activity(history, "plan", lambda: next(_PLAN_RESPONSES_2))
    record_activity(history, "post", lambda: post_slack("digest ready"),
                    idempotency_key=f"{run_id}:post")
    return plan


first_plan = run_workflow(history)              # original run
replay_plan = run_workflow(history)             # crash → replay with SAME history

print("original plan:", first_plan)
print("replay plan  :", replay_plan)
print("diverged?    :", first_plan != replay_plan, "(False → replay reused the recorded plan)")
print("slack posts  :", len(SIDE_EFFECTS), "(1 → side effect stayed exactly-once across replay)")
assert first_plan == replay_plan and len(SIDE_EFFECTS) == 1, "replay was not consistent/exactly-once"
print("PASS — deterministic on replay AND exactly-once side effects")

**What you just saw.** Same workflow, replayed — and this time it did *not* diverge: the recorded plan was returned instead of re-calling the drifting model, and the Slack post stayed at one because its `idempotency_key` found the recorded outcome. This is checkpointing (Ch 14) expressed as a workflow primitive: the LLM's stochasticity lives *outside* the deterministic shell, frozen inside recorded steps.

## Event-driven aside

Push decoupling further and components stop calling each other directly — they **emit events** (`document.indexed`, `agent.run.completed`) that interested parties subscribe to. You get loose coupling and a natural audit trail, at the cost of harder end-to-end tracing. The correctness machinery is exactly Ch 29's: reliable publishing via the **transactional outbox**, multi-step consistency via **sagas**, **idempotent consumers**, and the truth that *exactly-once is a myth* you engineer around — the same idempotency-key discipline you just applied to the recorded activity.

## 🎯 Senior lens

Two judgment calls live here. First: **know when *not* to hand-roll a resumable, multi-day workflow in Celery.** A queue is superb for fire-and-forget and periodic tasks; the moment work needs to wait for hours, pause for a human, and resume exactly where it left off, reach for the durable-execution engine built for it — bolting that onto Celery is a classic path to a fragile mess.

Second: **non-determinism is a property of the workload, not a bug to suppress.** An agent *will* answer differently on a re-call; the architecture's job is to capture that answer once and replay it, never to pretend the model is deterministic. Recorded activities + per-step idempotency keys are how you get durability and exactly-once effects *without* lying about what the model is.

## Recap

- **At-least-once → design for re-execution.** An **idempotency key** + an "already done?" **ledger** makes a side effect "effectively once" even when the task runs twice.
- **Match the tool to the work:** fire-and-forget/periodic → **Celery**; long, stateful, resumable with waits/human steps → **durable execution (Temporal)**; scheduled DAG of data transforms → **Prefect/Airflow**. (Dramatiq simpler, RQ Redis-only, arq asyncio.)
- **Durable execution = event-sourced replay** — re-run from the top, feeding recorded step results — which *assumes determinism*.
- **Agents break that assumption:** re-calling the LLM on replay returns a different answer, so live execution diverges from its recorded history.
- **The fix:** each model/tool call is a **recorded activity** (result persisted once, returned on replay) with an **idempotency key** per side effect → deterministic on replay *and* exactly-once.
- **Event-driven glue** (outbox/saga/idempotent consumers, Ch 29) keeps decoupled components correct.

## Exercises

Each *changes something* and asks you to predict the result first.

1. **Distinct keys, distinct effects.** Call `send_welcome_email_idempotent` for `user-7` with key `signup:user-7` and again with key `resend:user-7`. Predict how many emails land, then verify — and explain when a *new* key is the right call vs. a bug.
2. **Break replay on purpose.** In the `record_activity` demo, remove the `"plan"` recording (force the model to be re-called on replay). Predict whether the run diverges, then confirm, and describe the corruption it would cause downstream.
3. **Half-finished resume.** Pre-seed `EFFECT_LEDGER[f"{run_id}:post"]` before the first run (simulating a crash *after* the side effect but *before* recording the step). Predict how many Slack posts occur, then verify the idempotency key prevents a duplicate.
4. **Pick the tool.** For three workloads of your own — a webhook fan-out, a 5-day customer-onboarding flow with approvals, and a nightly feature-table rebuild — call `pick_tool(...)` and justify each in a markdown cell.

In [ ]:
# Exercise 1 — your code here.


In [ ]:
# Exercise 2 — your code here.


In [ ]:
# Exercise 3 — your code here.


## Next

- **Next notebook:** [`31-03-build-async-agent-runs-and-scheduled-automation.ipynb`](31-03-build-async-agent-runs-and-scheduled-automation.ipynb) — 🔧 the chapter's Build: `/runs` enqueues, a worker runs the loop with checkpoints, plus a scheduled "morning digest."
- **Book:** §31.4 (background jobs / long-running agent tasks), §31.5 + §31.5.1 (durable orchestration meets non-determinism), §31.7 (event-driven, briefly).
- **Blueprint this feeds:** [`blueprints/agent-loop/`](../../blueprints/agent-loop/) — its loop becomes an idempotent, recorded-activity workflow.
- **Capstone:** the idempotent-task patterns and the `record_activity` shape land in `capstone-project/workers/`; the ledger maps to a `UNIQUE`-keyed table in `capstone-project/` data (Ch 30).